In [1]:
import gc
import os
import re
import nltk
import spacy
import torch
import pickle
import joblib

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
# from datasets import Dataset
from collections import Counter
from nltk.data import load as nltk_load
from nltk.tokenize import PunktSentenceTokenizer
from typing import Callable, List, Tuple

from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
# from catboost import CatBoostClassifier, CatBoostRegressor

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelWithLMHead, AutoModelForCausalLM

/Users/t-andrew.widjaya/Documents/Thesis/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CROSS_ENTROPY = torch.nn.CrossEntropyLoss(reduction='none')
NLTK          = PunktSentenceTokenizer()
# DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE        = "cpu"

In [3]:
sent_cut_en = NLTK.tokenize

In [4]:
import torch
from transformers import PreTrainedTokenizerBase
from torch.nn.functional import cross_entropy

def gptneo_features(text, tokenizer: PreTrainedTokenizerBase, model, sent_cut):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    inputs = tokenizer(
        text,
        max_length=1024,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(DEVICE)

    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    shift_logits = logits[..., :-1, :].contiguous()
    shift_target = input_ids[..., 1:].contiguous()

    loss = cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_target.view(-1),
        reduction="none"
    ).view(shift_target.size())

    # Sentence-level perplexity calculations
    sentences = sent_cut(text)
    offsets = []
    token_ids = []
    for sent in sentences:
        tokens = tokenizer(sent, truncation=True, return_tensors="pt").input_ids.squeeze(0)
        offsets.append((len(token_ids), len(token_ids) + len(tokens)))
        token_ids.extend(tokens.tolist())

    sent_ppl = []
    for start, end in offsets:
        nll = loss[0, start:end].sum() / (end - start)
        sent_ppl.append(nll.exp().item())

    max_sent_ppl = max(sent_ppl)
    sent_ppl_avg = sum(sent_ppl) / len(sent_ppl)
    sent_ppl_std = torch.std(torch.tensor(sent_ppl)).item() if len(sent_ppl) > 1 else 0

    # Step-level perplexity calculations
    mask = torch.ones_like(loss[0]).to(DEVICE)
    step_ppl = loss[0].cumsum(dim=0) / mask.cumsum(dim=0)
    step_ppl = step_ppl.exp()
    max_step_ppl = step_ppl.max().item()
    step_ppl_avg = step_ppl.mean().item()
    step_ppl_std = step_ppl.std().item() if step_ppl.size(0) > 1 else 0

    # Overall perplexity
    text_ppl = loss.mean().exp().item()

    # Rank computation (GLTR metrics)
    all_probs = torch.softmax(shift_logits, dim=-1)
    rank_counts = [0, 0, 0, 0]
    for i in range(shift_target.size(1)):  # Iterate along sequence length
        token_probs = all_probs[0, i]
        target_token = shift_target[0, i]
        token_rank = (token_probs > token_probs[target_token]).sum().item()

        if token_rank < 10:
            rank_counts[0] += 1
        elif token_rank < 100:
            rank_counts[1] += 1
        elif token_rank < 1000:
            rank_counts[2] += 1
        else:
            rank_counts[3] += 1

    # Compile perplexity metrics
    ppls = [
        text_ppl, max_sent_ppl, sent_ppl_avg, sent_ppl_std,
        max_step_ppl, step_ppl_avg, step_ppl_std
    ]

    return rank_counts, ppls

In [5]:
cols = [
    'text_ppl', 'max_sent_ppl', 'sent_ppl_avg', 'sent_ppl_std', 'max_step_ppl', 
    'step_ppl_avg', 'step_ppl_std', 'rank_0', 'rank_10', 'rank_100', 'rank_1000'
]

final_cols = [
    'text',  
    'source',
    'prompt_id',
    'text_length' ,
    'word_count',
    'label', 'text_ppl', 'max_sent_ppl', 'sent_ppl_avg', 'sent_ppl_std', 'max_step_ppl', 
    'step_ppl_avg', 'step_ppl_std', 'rank_0', 'rank_10', 'rank_100', 'rank_1000'
]

In [7]:
for i in range(100,101):
    curr_num = i
    train          = pd.read_csv(f'Data/Chunked_Data_3/chunk_{curr_num}.csv')
    # train          = pd.read_csv(f'Chunked_Data_3/chunk_{curr_num}.csv')
    train['label'] = np.where(train['source'] == 'Human', 0, 1)
    train          = train.drop_duplicates(subset=['text']).reset_index(drop=True).dropna(subset=['text'])
    models_train_feats = []

    TOKENIZER_EN = AutoTokenizer.from_pretrained("gpt-neo-125m/tokenizer")
    MODEL_EN = AutoModelForCausalLM.from_pretrained("gpt-neo-125m/model").to(DEVICE)

    train_ppl_feats  = []
    train_gltr_feats = []
    print(f"Chunk {curr_num} In Process")
    with torch.no_grad():
        for text in tqdm(train.text.values):
            gltr, ppl = gptneo_features(text, TOKENIZER_EN, MODEL_EN, sent_cut_en)
            train_ppl_feats.append(ppl)
            train_gltr_feats.append(gltr)

    X_train = pd.DataFrame(
        np.concatenate((train_ppl_feats, train_gltr_feats), axis=1), 
        columns=[f'gpt-neo-125m-{col}' for col in cols]
    )
    models_train_feats.append(X_train)

    del X_train
    del TOKENIZER_EN; del MODEL_EN
    del train_ppl_feats; del train_gltr_feats

    gc.collect()
    torch.cuda.empty_cache()

    train_feats = pd.concat(models_train_feats, axis=1)
    train_feats = pd.concat([train,train_feats], axis = 1)
    train_feats.to_csv(f"Data/gpt-neo-125m-feature/Split3/chunk_{curr_num}.csv",index=False)
    print(f"Chunk {curr_num} extracted")

Chunk 100 In Process


100%|██████████| 494/494 [05:30<00:00,  1.49it/s]

Chunk 100 extracted
